# 🏠 House Price Prediction — Preprocessing
## Objectif
Nettoyer et préparer les données pour la modélisation :
valeurs manquantes, outliers, encodage et normalisation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Charger les données
df = pd.read_csv('../data/train.csv')
print("Données chargées :", df.shape)

Données chargées : (1460, 81)


In [2]:
# Supprimer les outliers détectés dans l'EDA
df = df[~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 200000))]
print("Taille après suppression outliers :", df.shape)

Taille après suppression outliers : (1458, 81)


In [3]:
# Transformation logarithmique du prix
df['SalePrice'] = np.log(df['SalePrice'])
print("Prix transformé - moyenne :", round(df['SalePrice'].mean(), 2))
print("Prix transformé - std :", round(df['SalePrice'].std(), 2))

Prix transformé - moyenne : 12.02
Prix transformé - std : 0.4


In [4]:
# Variables où NaN veut dire "n'existe pas"
cols_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'MasVnrType']

for col in cols_none:
    df[col] = df[col].fillna('None')

# Variables numériques où NaN veut dire 0
cols_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
             'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in cols_zero:
    df[col] = df[col].fillna(0)

# LotFrontage — on remplace par la médiane du quartier
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

# Electrical — 1 seul manquant, on met le plus fréquent
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

# Vérification
print("Valeurs manquantes restantes :", df.isnull().sum().sum())

Valeurs manquantes restantes : 0


# Feature Engineering

Création de variables dérivées qui capturent des aspects métier du prix immobilier, invisibles aux modèles à partir des colonnes brutes :

1. **Surfaces combinées & ratios** — `TotalSF`, `TotalBath`, `LivLotRatio`, …
2. **Âge & temporalité** — `HouseAge`, `RemodAge`, `IsNew`, `IsRemodeled`
3. **Flags binaires** — `HasPool`, `HasGarage`, `HasFireplace`, …
4. **Encodage ordinal** des variables de qualité (`Po < Fa < TA < Gd < Ex`)
5. **Interactions** qualité × surface
6. **Re-typage** des numériques qui sont en fait catégoriels (`MSSubClass`, `MoSold`, `YrSold`)
7. **Correction du skew** des variables numériques (`log1p`)

In [5]:
# === 2. Surfaces combinées et ratios ===
# Surface totale = sous-sol + RDC + étage
df['TotalSF']      = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
df['TotalLivArea'] = df['GrLivArea']   + df['TotalBsmtSF']

# Somme de toutes les terrasses / porches
df['TotalPorchSF'] = (df['OpenPorchSF'] + df['EnclosedPorch'] +
                      df['3SsnPorch']   + df['ScreenPorch']   + df['WoodDeckSF'])

# Nombre total de salles de bain pondéré (half = 0.5)
df['TotalBath'] = (df['FullBath']     + 0.5 * df['HalfBath'] +
                   df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath'])

# Ratios
df['LivLotRatio']  = df['GrLivArea'] / df['LotArea']
df['BsmtFinRatio'] = (df['BsmtFinSF1'] + df['BsmtFinSF2']) / df['TotalBsmtSF'].replace(0, 1)

new = ['TotalSF', 'TotalLivArea', 'TotalPorchSF', 'TotalBath', 'LivLotRatio', 'BsmtFinRatio']
print(df[new].describe().round(2))

       TotalSF  TotalLivArea  TotalPorchSF  TotalBath  LivLotRatio  \
count  1458.00       1458.00       1458.00    1458.00      1458.00   
mean   2557.15       2563.00        180.81       2.21         0.18   
std     774.11        776.19        156.12       0.78         0.11   
min     334.00        334.00          0.00       1.00         0.01   
25%    2008.50       2014.00         45.00       2.00         0.12   
50%    2473.00       2476.50        164.00       2.00         0.16   
75%    3002.25       3005.50        265.00       2.50         0.20   
max    6872.00       6872.00       1027.00       6.00         0.95   

       BsmtFinRatio  
count       1458.00  
mean           0.44  
std            0.36  
min            0.00  
25%            0.00  
50%            0.50  
75%            0.76  
max            1.00  


In [6]:
# === 3. Features temporelles (âge) ===
df['HouseAge']  = df['YrSold'] - df['YearBuilt']
df['RemodAge']  = df['YrSold'] - df['YearRemodAdd']
df['GarageAge'] = df['YrSold'] - df['GarageYrBlt']
df.loc[df['GarageYrBlt'] == 0, 'GarageAge'] = 0   # pas de garage → âge neutre

df['IsNew']         = (df['YearBuilt']    == df['YrSold']).astype(int)
df['IsRemodeled']   = (df['YearBuilt']    != df['YearRemodAdd']).astype(int)
df['IsRecentRemod'] = (df['YrSold'] - df['YearRemodAdd'] < 5).astype(int)

print(f"Âge moyen des maisons : {df['HouseAge'].mean():.1f} ans")
print(f"% maisons neuves      : {df['IsNew'].mean()*100:.1f} %")
print(f"% maisons rénovées    : {df['IsRemodeled'].mean()*100:.1f} %")

Âge moyen des maisons : 36.6 ans
% maisons neuves      : 4.3 %
% maisons rénovées    : 47.7 %


In [7]:
# === 4. Flags binaires : présence d'une caractéristique ===
df['HasPool']      = (df['PoolArea']     > 0).astype(int)
df['HasGarage']    = (df['GarageArea']   > 0).astype(int)
df['HasBsmt']      = (df['TotalBsmtSF']  > 0).astype(int)
df['HasFireplace'] = (df['Fireplaces']   > 0).astype(int)
df['Has2ndFloor']  = (df['2ndFlrSF']     > 0).astype(int)
df['HasPorch']     = (df['TotalPorchSF'] > 0).astype(int)
df['HasMasVnr']    = (df['MasVnrArea']   > 0).astype(int)
df['HasShed']      = (df['MiscFeature']  == 'Shed').astype(int)

flags = ['HasPool','HasGarage','HasBsmt','HasFireplace',
         'Has2ndFloor','HasPorch','HasMasVnr','HasShed']
print("Taux de possession :")
print((df[flags].mean() * 100).round(1).astype(str) + ' %')

Taux de possession :
HasPool          0.4 %
HasGarage       94.4 %
HasBsmt         97.5 %
HasFireplace    52.7 %
Has2ndFloor     43.1 %
HasPorch        82.6 %
HasMasVnr       40.4 %
HasShed          3.4 %
dtype: object


In [8]:
# === 5. Encodage ordinal des variables de qualité ===
# L'ordre Po < Fa < TA < Gd < Ex a un vrai sens : on le conserve numériquement
qual_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

for c in qual_cols:
    df[c + '_ord'] = df[c].map(qual_map).fillna(0).astype(int)

# Autres variables ordinales spécifiques
df['BsmtExposure_ord'] = df['BsmtExposure'].map(
    {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}).fillna(0).astype(int)

df['BsmtFinType1_ord'] = df['BsmtFinType1'].map(
    {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
).fillna(0).astype(int)

df['GarageFinish_ord'] = df['GarageFinish'].map(
    {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}).fillna(0).astype(int)

df['Functional_ord'] = df['Functional'].map(
    {'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3, 'Mod': 4,
     'Min2': 5, 'Min1': 6, 'Typ': 7}).fillna(7).astype(int)

print(f"✓ {len(qual_cols) + 4} features ordinales créées")

✓ 14 features ordinales créées


In [9]:
# === 6. Interactions entre qualité et surface/équipement ===
df['OverallScore'] = df['OverallQual'] * df['OverallCond']
df['QualSF']       = df['OverallQual'] * df['TotalSF']
df['QualBath']     = df['OverallQual'] * df['TotalBath']
df['QualGarage']   = df['OverallQual'] * df['GarageArea']
df['ExterScore']   = df['ExterQual_ord']  * df['ExterCond_ord']
df['BsmtScore']    = df['BsmtQual_ord']   * df['BsmtCond_ord']
df['GarageScore']  = df['GarageQual_ord'] * df['GarageCond_ord'] * df['GarageArea']

print("Corrélations avec SalePrice :")
print(df[['OverallScore','QualSF','QualBath','QualGarage','GarageScore','SalePrice']]
      .corr()['SalePrice'].drop('SalePrice').round(3))

Corrélations avec SalePrice :
OverallScore    0.608
QualSF          0.883
QualBath        0.830
QualGarage      0.799
GarageScore     0.641
Name: SalePrice, dtype: float64


In [10]:
# === 7. Re-typage : certaines variables numériques sont en fait catégorielles ===
# MSSubClass est un code de type de logement, pas une valeur ordonnée
# MoSold / YrSold → on évite l'effet ordinal trompeur
df['MSSubClass'] = df['MSSubClass'].astype(str)
df['MoSold']     = df['MoSold'].astype(str)
df['YrSold']     = df['YrSold'].astype(str)

print("Types re-castés :", df[['MSSubClass', 'MoSold', 'YrSold']].dtypes.to_dict())

Types re-castés : {'MSSubClass': dtype('O'), 'MoSold': dtype('O'), 'YrSold': dtype('O')}


In [11]:
# === 8. Correction de l'asymétrie (skew) des variables numériques ===
from scipy.stats import skew

num_feats = df.select_dtypes(include=[np.number]).columns.drop('SalePrice')
skewness = df[num_feats].apply(lambda x: skew(x.dropna()))
skewed_feats = skewness[abs(skewness) > 0.75].index.tolist()

print(f"Nombre de features asymétriques (|skew| > 0.75) : {len(skewed_feats)}")
for c in skewed_feats:
    df[c] = np.log1p(df[c])

print("→ Transformation log1p appliquée.")
print("Shape finale avant encodage :", df.shape)

Nombre de features asymétriques (|skew| > 0.75) : 43
→ Transformation log1p appliquée.
Shape finale avant encodage : (1458, 122)


In [12]:
# One-Hot Encoding des variables catégorielles
df = pd.get_dummies(df, drop_first=True)
print("Taille après encodage :", df.shape)

Taille après encodage : (1458, 327)


In [13]:
# Séparer features et target
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']
print("X shape :", X.shape)

X shape : (1458, 326)


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
print("Features shape :", X_scaled.shape)

Features shape : (1458, 326)


In [15]:
import os

# Créer le dossier processed
os.makedirs('../data/processed', exist_ok=True)

# Sauvegarder les données preprocessées
X_df = pd.DataFrame(X_scaled, columns=X.columns)
X_df.to_csv('../data/processed/X_scaled.csv', index=False)
y.to_csv('../data/processed/y.csv', index=False)

print("Données sauvegardées !")

Données sauvegardées !


In [16]:
import joblib
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(list(X.columns), '../models/columns.pkl')
joblib.dump(skewed_feats, '../models/skewed_feats.pkl')
print("Sauvegardé !")

Sauvegardé !
